In [74]:
from google.colab import userdata
import os

hf_token = userdata.get("HF_TOKEN")
weather_api_key = userdata.get("WEATHER_API_KEY")

os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token
os.environ["WEATHER_API_KEY"] = weather_api_key

In [75]:
!pip install -q langchain langchain-core langchain-community pydantic duckduckgo-search ddgs langchain-experimental requests langchain-huggingface

In [76]:
from langchain_core.tools import tool
import requests

In [77]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool=DuckDuckGoSearchRun()

In [78]:
@tool
def get_weather_data(city:str) -> str:
  """
  This function fetches the current weather data for a given city
  """
  url = f'https://api.weatherstack.com/current?access_key={weather_api_key}&query={city}'

  response=requests.get(url)

  return response.json()

In [79]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
)

llm = ChatHuggingFace(llm=llm)

In [80]:
# step-1: ReAct prompt

from langchain_core.prompts import PromptTemplate

prompt=PromptTemplate(
    template="""Answer the following questions as best you can. You have access to the following tools:

                {tools}

                Use the following format:

                Question: the input question you must answer
                Thought: you should always think about what to do
                Action: the action to take, should be one of [{tool_names}]
                Action Input: the input to the action in valid JSON format
                Observation: the result of the action
                ... (this Thought/Action/Action Input/Observation can repeat N times)
                Thought: I now know the final answer
                Final Answer: the final answer to the original input question

                Do not generate Observation yourself. Observation will be provided by the executor after the tool runs.

                Begin!

                Question: {input}
                {agent_scratchpad}
                """,
    input_variables=["input", "agent_scratchpad", "tools", "tool_names"]
)

In [81]:
from abc import ABC, abstractmethod

class Runnable(ABC):
  @abstractmethod
  def invoke(self,input_data):
    pass

In [82]:
# React Output Parser

import re
import json

class ReActOutputParser:

    def parse(self, agent_output):

        # Case 1: final answer
        if "Final Answer:" in agent_output:
            final_answer = agent_output.split("Final Answer:")[-1].strip()

            return {
                "type": "finish",
                "final_answer": final_answer
            }

        # Case 2: action
        action_match = re.search(
            r"Action\s*:\s*(.*)",
            agent_output
        )

        action_input_match = re.search(
            r"Action Input\s*:\s*(.*)",
            agent_output
        )

        if not action_match or not action_input_match:
            raise ValueError(f"Could not parse agent output:\n{agent_output}")

        action = action_match.group(1).strip()
        action_input = action_input_match.group(1).strip()

        # convert JSON string to Python dict
        try:
            action_input = json.loads(action_input)
        except:
            # fallback if model gives plain string instead of JSON
            action_input = action_input

        return {
            "type": "action",
            "action": action,
            "action_input": action_input
        }

In [83]:
# ReAct Agent
class ReActAgent(Runnable):

  def __init__(self,llm,tools,prompt):
    self.llm=llm
    self.tools=tools
    self.prompt=prompt

  def invoke(self,input,agent_scratchpad):
    # convert tools to text
    tool_text="\n".join([f"{tool.name}: {tool.description}" for tool in self.tools])
    tool_names=", ".join([tool.name for tool in self.tools])
    chain=self.prompt|self.llm
    response=chain.invoke({
        "input":input,
        "agent_scratchpad":agent_scratchpad,
        "tools":tool_text,
        "tool_names":tool_names
    })
    return response.content

In [84]:
# Agent Executor
class AgentExecutor(Runnable):

  def __init__(self,agent,tools,max_iter=10):
    self.agent=agent
    self.tools=tools
    self.max_iter=max_iter
    self.parser=ReActOutputParser()

    self.tool_map={
        tool.name:tool for tool in self.tools
    }

  def invoke(self,input):
    agent_scratchpad="Thought: "
    for step in range(self.max_iter):
      agent_output=self.agent.invoke(input=input,agent_scratchpad=agent_scratchpad)
      parsed_output=self.parser.parse(agent_output)

      if parsed_output["type"]=="finish":
        return parsed_output["final_answer"]

      if parsed_output["type"]=="action":
        action=parsed_output["action"]
        action_input=parsed_output["action_input"]
        if action not in self.tool_map:
                observation = f"Invalid tool: {action}"
        else:
                tool = self.tool_map[action]
                observation = tool.invoke(action_input)
      agent_scratchpad += f"""{agent_output}
                          Observation: {observation}
                          Thought: """
    return "Agent stopped because max iterations were reached."


In [85]:
agent=ReActAgent(
    llm=llm,
    tools=[search_tool,get_weather_data],
    prompt=prompt
    )

In [86]:
agent_executor=AgentExecutor(
    agent=agent,
    tools=[search_tool,get_weather_data],
    max_iter=10
)

In [87]:
# invoke
response = agent_executor.invoke("Find the capital of Madhya Pradesh, then find it's current weather condition")

In [88]:
response

'The capital of Madhya Pradesh is Bhopal, and the current weather conditions in Bhopal are sunny with a temperature of 32°C, humidity of 68%, and wind speed of 5 km/h.'